## 1. What prompt engineering actually is

**Prompt engineering is the practice of iteratively improving prompts to get more reliable, higher-quality outputs from an LLM.**

Two things this isn't:
- ❌ "Writing one perfect prompt" — prompt engineering is a loop, not a one-shot
- ❌ "A bag of tricks" — techniques are tools applied selectively based on what eval data says

---

## 2. The iterative improvement loop

```
┌──────────────────────────────────────────────────┐
│  Set goal: what should the prompt accomplish?    │
└────────────────────┬─────────────────────────────┘
                     ↓
┌──────────────────────────────────────────────────┐
│  Write initial prompt (v1, naive)                │
└────────────────────┬─────────────────────────────┘
                     ↓
┌──────────────────────────────────────────────────┐
│  Evaluate against test cases                     │
└────────────────────┬─────────────────────────────┘
                     ↓
┌──────────────────────────────────────────────────┐
│  Apply technique (CoT / few-shot / XML / etc.)   │
└────────────────────┬─────────────────────────────┘
                     ↓
┌──────────────────────────────────────────────────┐
│  Re-evaluate — did it actually improve?          │
└────────────────────┬─────────────────────────────┘
                     ↓ measurable improvement
                  Repeat
```

**Key insight**: Steps 3 and 5 ("evaluate") are what separate prompt engineering from prompt writing. Without them, you're just guessing that your changes helped.

---

## 3. System vs. user messages — where does content go?

**Rule of thumb**: **static** content → `system`; **dynamic** content → `messages`.

| Goes in `system` (set once, rarely changes) | Goes in `messages` (per-call) |
|---|---|
| Role / persona | The actual user input for this turn |
| Behavioral rules (cite sources, no fabrication) | Conversation history |
| Output format spec (JSON schema) | Dynamically retrieved RAG context |
| Tone and style guidance | Task-specific inputs (the code to review, the sentence to translate) |
| Domain constraints (only X topics) | Prefills (assistant-role messages to steer output) |
| Static reference content (company info, product docs) | Few-shot examples (when they vary per query) |
| Few-shot examples (when they're stable) | |

**Why this split matters**:

- **Prompt caching** (Phase 4) caches the `system` prefix — putting static content there saves cost on every call
- **Multi-turn coherence** — keeping role in `system` means it persists naturally across the conversation without re-injection
- **Debugging** — when something goes wrong, you know exactly which section to modify

---

## 4. Core techniques (when to use what)

### (a) Clear & direct instructions

The simplest technique — give the LLM the most precise instruction possible. Anti-patterns:

- ❌ Vague: "Help with this code"
- ❌ Phrased as a question: "Can you write a function that...?"
- ❌ Hedged: "Maybe try to..."

Better:

- ✅ Specific: "Write a Python function that validates AWS S3 bucket names according to AWS rules (3-63 chars, lowercase, no consecutive hyphens). Return True/False."
- ✅ Direct verb start: "Write" / "Generate" / "Identify" / "Compare"
- ✅ State the goal upfront: "Your task is X. Output format is Y."

**When this alone is enough**: simple factual queries, well-defined single-step tasks. Don't reach for fancier techniques first.

### (b) Few-shot / multi-shot examples

Provide 2-5 examples to teach format and reasoning by demonstration. Almost always paired with XML tags for structure:

```xml
<example_1>
  <input>banana, 1 medium</input>
  <output>{"name": "banana", "calories": 105, "protein_g": 1.3}</output>
</example_1>

<example_2>
  <input>grilled chicken breast, 150g</input>
  <output>{"name": "grilled chicken breast", "calories": 248, "protein_g": 46.5}</output>
</example_2>

<input>brown rice, 200g</input>
```

**Design decisions**:
- **Example count**: 2-3 for simple tasks, 4-5 for complex formats. More isn't always better — diminishing returns + context cost
- **Example selection**: include edge cases (mixed dishes, unusual portions), not just easy ones
- **Example ordering**: recency bias is real — put the most representative example last

**When to use**: format consistency matters; the task has a learnable pattern; zero-shot gives inconsistent structure.

### (c) Chain-of-Thought (CoT)

Force the LLM to externalize its reasoning before answering:

```
"Analyze this meal and compute remaining daily calorie budget.
Think step by step:
1. Identify each food and its portion
2. Look up nutritional data for each
3. Sum to get meal total
4. Compare against daily target
5. Then give recommendation"
```

**When CoT helps**: multi-step reasoning, math, planning, disambiguation tasks where intermediate steps matter.

**When CoT hurts**: simple factual queries. Adding "think step by step" to "What's 2+2?" wastes tokens and can actually reduce accuracy on trivial tasks (because the LLM overthinks).

### (d) XML tags + structured output

Two distinct uses of XML, often confused:

**XML for input structure** (organizing the prompt itself):
```xml
<context>User is on a low-sodium diet</context>
<task>Analyze this meal photo</task>
<constraints>
- Flag any sodium > 600mg per serving
- Use g for solids, ml for liquids
</constraints>
```

**XML for output structure** (forcing parseable response):
```
"Respond inside <result>...</result> tags, with this JSON schema: ..."
```

**Why XML beats plain text**: Claude is specifically trained to honor XML boundaries — instructions inside `<safety>` tags are treated differently from instructions inside `<task>`. With plain text, the LLM has to infer where one instruction ends and the next begins.

---

<Interview Q&A — Prompt Engineering>

### Q1: What's the iterative improvement loop?

Set goal → write initial prompt → evaluate against test cases → apply a technique → re-evaluate → repeat. **The evaluation steps are the non-skippable parts**. Without them, you're guessing whether your changes helped. Prompt engineering without eval is prompt writing.

### Q2: How do you decide what goes in `system` vs `messages`?

**Static content → `system`. Dynamic content → `messages`.**

- `system`: role, behavioral rules, output format, tone, domain constraints, stable reference content, stable few-shot examples
- `messages`: per-turn user input, conversation history, RAG-retrieved context, task-specific inputs (code to review, image to classify), prefills

**Three reasons this split matters**:
1. **Prompt caching** — Anthropic caches the `system` prefix; putting static content there gives ~90% cost reduction on repeated calls
2. **Coherence** — system content persists across multi-turn conversations without re-injection
3. **Debuggability** — clear separation of "what doesn't change" vs "what's request-specific"

### Q3: When do you use few-shot vs zero-shot?

**Zero-shot**: simple factual queries, well-defined tasks the LLM already knows how to do well. No examples needed.

**Few-shot**: when format consistency matters, when the task has a learnable pattern the LLM doesn't get right zero-shot, when edge cases need to be demonstrated. Typically 2-5 examples, wrapped in XML tags.

**Practical signal**: if zero-shot gives 80%+ accuracy on your eval set, don't bother with few-shot. If it gives <60%, few-shot is the first thing to try.

### Q4: When does Chain-of-Thought help vs hurt?

**Helps**: multi-step reasoning, math, planning, ambiguity resolution, any task where intermediate steps matter to the answer.

**Hurts**: simple factual lookups (wastes tokens), trivial tasks (LLM overthinks), real-time interactive UX (latency cost).

**Heuristic**: if a human would naturally write down their work to solve this task, CoT will help the LLM too. If a human would answer in one breath, CoT won't.

### Q5: Why XML tags for prompts (specifically with Claude)?

Three reasons:
1. **Claude is trained to honor XML boundaries** — sections inside `<task>`, `<constraints>`, `<safety>` tags are treated as separate contracts. Plain text forces the LLM to infer boundaries.
2. **Debugging** — when output format is wrong, you know to modify the `<output_format>` block. With plain text, you re-read the whole prompt to find what to change.
3. **Layered modification** — XML lets you swap one section (e.g., update the role definition) without affecting others. Critical for production prompt iteration.

</Interview Q&A — Prompt Engineering>